# Converting models and outputs to GEE usable formats

In [41]:
from geemap import ml
import joblib
import ee

In [42]:
ee.Initialize()

In [52]:
# Define source of the models to be uploaded to GEE
sensor_code = "S2"
class_code = "kbrgm"
plot_output_dir = f"C:/Users/s4770224/Documents/Work/Writing/Figures/Obj2/unmixing/simpleTrees/{sensor_code}/{class_code}/"

reload_dict = {
    "classifiers" : plot_output_dir + "classifiers.joblib",
    "regressors" : plot_output_dir + "regressors.joblib",
    "water_regressor": plot_output_dir + "water_regressor.joblib"}

s2_bands = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A"]
water_regressor_pca_comps = ["0", "1", "2", "3", "4", "5", "6", "7"]
pca_comps = ['0', '1', '2', '3', '4', '5', '6', '7', 'water']

In [49]:
# Define conversion and export function
def convertToGEEFormat(model, asset_id, band_names= pca_comps, mode = "Regression", verbose=True, debug = False, test = False):
    # Convert regressor to strings and then strings to E classifier. 
    model_str = ml.rf_to_strings(model, band_names, output_mode= mode)
    if verbose: 
        print(f"Model loaded with {len(model_str)} trees having ~{len(model_str[0])} nodes")
    
    if debug:
        print(len(model_str))          # how many trees were converted
        print(model_str[0][:1000])     # inspect the first tree
    if not test: 
        ml.export_trees_to_fc(model_str, asset_id) # Exports to featureClass at the identified asset path
    return

In [44]:
classifiers = joblib.load(reload_dict["classifiers"])
regressors = joblib.load(reload_dict["regressors"])
water_regressor = joblib.load(reload_dict["water_regressor"])

In [ ]:
# Export classifiers
for key in classifiers.keys():
    asset_name = f"projects/kelp-444901/assets/{key}_RFC_PCA_trees"
    convertToGEEFormat(classifiers[key], asset_name, pca_comps, mode = "CLASSIFICATION", verbose = True, test = True)

Model loaded with 100 trees having ~25583 nodes
Model loaded with 100 trees having ~26938 nodes
Model loaded with 100 trees having ~24628 nodes
Model loaded with 100 trees having ~30846 nodes
Model loaded with 100 trees having ~23678 nodes


In [ ]:
# Export regressors
for key in regressors.keys():
    asset_name = f"projects/kelp-444901/assets/{key}_RFR_PCA_trees"
    convertToGEEFormat(regressors[key], asset_name, pca_comps, mode = "REGRESSION", verbose = True, test = True)

Model loaded with 20 trees having ~32885 nodes
Model loaded with 20 trees having ~35782 nodes
Model loaded with 20 trees having ~36001 nodes
Model loaded with 20 trees having ~47318 nodes
Model loaded with 20 trees having ~41515 nodes


In [ ]:
# Export water regressor
asset_name = f"projects/kelp-444901/assets/water_RFR_PCA_trees"
convertToGEEFormat(water_regressor, asset_name, water_regressor_pca_comps, mode = "REGRESSION", verbose = True, test = True)

Model loaded with 20 trees having ~64750 nodes
